# Final Research Pipeline Colab
Notebook final untuk eksekusi pipeline penelitian dari Step 1 sampai Step 11 menggunakan `data/dataset_final.csv`.

## 0) Setup Runtime
Pilih GPU T4 di Colab: Runtime > Change runtime type > T4 GPU.

In [ ]:
import os
HF_TOKEN = 'hf_xxx_your_token_here'  # placeholder
os.environ['HF_TOKEN'] = HF_TOKEN

!git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment
!git checkout main
!python3 -m pip install -r requirements.txt
!nvidia-smi

## 1) Konfigurasi Final
Model config final: `src/resources/step7_final_production.json`
Trial: `final_best_basep1_finetune1_len96_bs4_h128_ce`

In [ ]:
import os
assert os.path.exists('data/dataset_final.csv'), 'Dataset final tidak ditemukan di repo'
assert os.path.exists('src/resources/step7_final_production.json'), 'Config final tidak ditemukan'
print('Dataset:', 'data/dataset_final.csv')
print('Config :', 'src/resources/step7_final_production.json')

## 2) Run Pipeline Step 1-11 (End-to-End)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
!python3 run_pipeline_full.py \
  --from-step 1 \
  --until-step 11 \
  --dataset-source final \
  --step7-config-json src/resources/step7_final_production.json \
  --step7-max-trials 1 \
  --run-class-multiplier

## 3) Tampilkan Hasil Utama

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

metrics = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
report = pd.read_csv('outputs/classification_report.csv')
cm = pd.read_csv('outputs/confusion_matrix.csv', index_col=0)
runlog = json.load(open('outputs/pipeline_run_log.json', encoding='utf-8'))

print('Pipeline source:', runlog.get('dataset_source'))
display(pd.DataFrame([metrics])[['accuracy','precision_macro','recall_macro','f1_macro']])
display(report)
display(cm)
if os.path.exists('outputs/figures/confusion_matrix.png'):
    display(Image(filename='outputs/figures/confusion_matrix.png'))

## 4) Artifact Final
- `outputs/pipeline_run_log.json`
- `outputs/final_metrics.json`
- `outputs/classification_report.csv`
- `outputs/confusion_matrix.csv`
- `outputs/error_analysis_summary.md`
- `outputs/bab4_hasil_otomatis.md`
- `outputs/bab5_kesimpulan_saran_otomatis.md`